In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
import sklearn
from sklearn import datasets
sklearn.set_config(display='text')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler, Normalizer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score

## 앙상블 보팅 알고리즘에 사용할 개별 모델을 만들기 위해서 import 한다.

In [2]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

## 보팅(voting)

여러 개의 분류 모델의 결과를 대상으로 투표를 통해서 최종 클래스의 레이블을 예측하는 방법이다.

분류기가 10개 있다고 했을 때, 특정 데이터에 대해 7개의 분류기는 클래스1이라고 예측하고, 나머지 3개의 분류기는 클래스2라고 예측했을 때 클래스1이 가장 높은 득표수를 보이므로 최종적으로 클래스1로 예측하는 것이다. 이를 다수결 투표라고 하는데 이와 비슷한 방법을 다수결이 아닌 절반 이상의 분류기의 표를 얻어야 하는 과반수 투표 방식이 있다.

개별 분류기로 지도 학습 방법 중 k최근접 이웃, 서포트 벡터 머신, 의사결정 트리, 나이브 베이즈 등 여러 알고리즘을 사용해서 다양한 분류 모델을 사용할 수 있다.

## 와인 데이터를 앙상블(보팅) 알고리즘에 사용할 데이터를 준비한다.

In [3]:
raw_data = datasets.load_wine()

xData = raw_data.data
yData = raw_data.target

x_train, x_test, y_train, y_test = train_test_split(xData, yData, random_state=0)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

## 보팅 모델을 만들고 학습 시킨다.

보팅 알고리즘을 사용하기 위해 VotingClassifier를 import 한다.

In [4]:
from sklearn.ensemble import VotingClassifier

보팅 알고리즘에 사용할 개별 머신러닝 모델 분류기를 만든다.

In [5]:
model_kn = KNeighborsClassifier()
model_sv = SVC(probability=True)
model_dt = DecisionTreeClassifier(random_state=0, max_depth=4)
model_nb = GaussianNB()

위에서 만든 개별 분류기 모델을 이용해서 보팅 모델을 만든다.

`estimators`에 보팅 알고리즘에 사용할 개별 분류기 모델들을 리스트로 묶어서 지정한다.  
`voting`에 투표 방식을 지정한다. 기본값은 hard이고 과반수 투표 방식을 사용하고 soft는 다수결 투표 방식을 사용한다.  
hard로 지정하면 predict_proba() 메소드를 사용할 수 없고 soft로 지정하면 사용할 수 있다.  
`weights`에 각 개별 분류기에 가중치를 지정해서 가중치 투표를 할 수 있다.

In [9]:
model = VotingClassifier(
    estimators=[('kn', model_kn), ('sv', model_sv), ('dt', model_dt), ('nb', model_nb)], voting='soft', weights=[1, 1, 1, 1]
)
model.fit(x_train, y_train)

VotingClassifier(estimators=[('kn', KNeighborsClassifier()),
                             ('sv', SVC(probability=True)),
                             ('dt',
                              DecisionTreeClassifier(max_depth=4,
                                                     random_state=0)),
                             ('nb', GaussianNB())],
                 voting='soft', weights=[1, 1, 1, 1])

## 학습된 보팅 모델을 테스트 데이터로 예측한다.

In [10]:
predict = model.predict(x_test)
print(predict)

[0 2 1 0 1 1 0 2 1 1 2 2 0 1 2 1 0 0 2 0 1 0 0 1 1 1 1 1 1 2 0 0 1 0 0 0 2
 1 1 2 0 0 1 1 1]


In [11]:
predict_proba = model.predict_proba(x_test)
print(predict_proba)

[[9.97245769e-01 1.31510434e-03 1.43912691e-03]
 [3.13399395e-03 5.11139258e-03 9.91754613e-01]
 [6.17499977e-03 9.91194590e-01 2.63041018e-03]
 [9.90020949e-01 7.92512718e-03 2.05392334e-03]
 [1.29793551e-01 8.61052332e-01 9.15411676e-03]
 [8.13406459e-02 9.06561251e-01 1.20981030e-02]
 [9.96672758e-01 1.23693892e-03 2.09030307e-03]
 [1.98498574e-03 2.50577593e-03 9.95509238e-01]
 [2.15914133e-03 9.97088601e-01 7.52257859e-04]
 [9.75419492e-04 9.96873902e-01 2.15067874e-03]
 [5.45488792e-03 2.03159806e-02 9.74229131e-01]
 [6.18930076e-03 2.74948071e-02 9.66315892e-01]
 [9.97988185e-01 8.93464996e-04 1.11834998e-03]
 [2.63209741e-01 7.33492220e-01 3.29803947e-03]
 [2.21351398e-03 2.85730635e-03 9.94929180e-01]
 [6.67338021e-04 9.98570281e-01 7.62381398e-04]
 [9.63256039e-01 3.11293126e-02 5.61464891e-03]
 [9.84896924e-01 9.97633887e-03 5.12673725e-03]
 [2.64285702e-03 3.21417924e-01 6.75939219e-01]
 [9.97693829e-01 1.03690543e-03 1.26926579e-03]
 [3.60947787e-01 6.32974554e-01 6.077659

## 모델을 평가한다.

In [12]:
classification = classification_report(y_test, predict, target_names=raw_data.target_names)
print(classification)

              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        16
     class_1       1.00      0.95      0.98        21
     class_2       0.89      1.00      0.94         8

    accuracy                           0.98        45
   macro avg       0.96      0.98      0.97        45
weighted avg       0.98      0.98      0.98        45

